# 🔍 Data Exploration - Retail Video Analytics

Notebook khám phá và kiểm tra data trong lakehouse (MinIO):
- **Bronze Layer**: Raw data từ Pulsar
- **Silver Layer**: Cleaned detections
- **Gold Layer**: Aggregated metrics

---

## 1️⃣ Setup & Connection

In [1]:
import pandas as pd
import trino
from datetime import datetime, timedelta
import json

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', None)

print("✅ Libraries loaded successfully")

✅ Libraries loaded successfully


In [2]:
# Connect to Trino (gateway to Iceberg data in MinIO)
conn = trino.dbapi.connect(
    host='localhost',
    port=8083,
    user='hungfnguyen',
    catalog='lakehouse',
    schema='rva'
)

cursor = conn.cursor()
print(f"✅ Connected to Trino")
print(f"   Catalog: {conn.catalog}")
print(f"   Schema: {conn.schema}")

✅ Connected to Trino
   Catalog: lakehouse
   Schema: rva


In [3]:
# Helper function for quick queries
def query(sql, show_query=True):
    """Execute SQL and return DataFrame"""
    if show_query:
        print(f"🔍 Query: {sql[:100]}..." if len(sql) > 100 else f"🔍 Query: {sql}")
    try:
        df = pd.read_sql(sql, conn)
        print(f"✅ Returned {len(df)} rows")
        return df
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

print("✅ Helper function ready")

✅ Helper function ready


---
## 2️⃣ Bronze Layer Exploration

**Bronze = Raw data** từ Pulsar topics, chứa JSON event từ Vision module

In [4]:
# List all tables
df_tables = query("SHOW TABLES FROM lakehouse.rva")
df_tables

🔍 Query: SHOW TABLES FROM lakehouse.rva


/tmp/ipykernel_118446/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 5 rows


,Table
0,bronze_raw
1,gold_hour_by_cam
2,gold_minute_by_cam
3,gold_track_summary
4,silver_detections


In [5]:
# Describe Bronze table schema
df_schema = query("DESCRIBE lakehouse.rva.bronze_raw")
df_schema

🔍 Query: DESCRIBE lakehouse.rva.bronze_raw


/tmp/ipykernel_118446/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 7 rows


,Column,Type,Extra,Comment
0,schema_version,varchar,,
1,pipeline_run_id,varchar,,
2,frame_index,bigint,,
3,payload,varchar,,
4,camera_id,varchar,,
5,store_id,varchar,,
6,ingest_ts,timestamp(6),,


In [6]:
# Count total records in Bronze
df_count = query("SELECT COUNT(*) as total_records FROM lakehouse.rva.bronze_raw")
total = df_count.iloc[0]['total_records']
print(f"\n📊 Total Bronze records: {total:,}")
df_count

🔍 Query: SELECT COUNT(*) as total_records FROM lakehouse.rva.bronze_raw


/tmp/ipykernel_118446/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 1 rows

📊 Total Bronze records: 514


,total_records
0,514


In [7]:
# Check latest data timestamp
df_latest = query("""
SELECT 
    MIN(ingest_ts) as earliest,
    MAX(ingest_ts) as latest,
    CURRENT_TIMESTAMP as now,
    CAST((CURRENT_TIMESTAMP - MAX(ingest_ts)) AS VARCHAR) as data_lag
FROM lakehouse.rva.bronze_raw
""")
print("\n⏰ Data freshness:")
df_latest

🔍 Query: 
SELECT 
    MIN(ingest_ts) as earliest,
    MAX(ingest_ts) as latest,
    CURRENT_TIMESTAMP as now,...


/tmp/ipykernel_118446/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 1 rows

⏰ Data freshness:


,earliest,latest,now,data_lag
0,2025-12-06 15:56:39.481,2025-12-06 15:57:19.069,2025-12-06 15:57:48.347000+00:00,0 07:00:29.278


In [8]:
# Sample 5 raw events
df_sample = query("""
SELECT 
    store_id,
    pipeline_run_id,
    camera_id,
    frame_index,
    ingest_ts,
    json_extract_scalar(payload, '$.frame_index') as vision_frame_idx,
    json_array_length(json_extract(payload, '$.detections')) as num_detections
FROM lakehouse.rva.bronze_raw
ORDER BY ingest_ts DESC
LIMIT 5
""")
print("\n📋 Latest 5 events:")
df_sample

🔍 Query: 
SELECT 
    store_id,
    pipeline_run_id,
    camera_id,
    frame_index,
    ingest_ts,
    json_...


/tmp/ipykernel_118446/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 5 rows

📋 Latest 5 events:


,store_id,pipeline_run_id,camera_id,frame_index,ingest_ts,vision_frame_idx,num_detections
0,store_01,c5c81de4a1a84d6eb3770f23367fbc30,cam_01,514,2025-12-06 15:57:19.069,514,6
1,store_01,c5c81de4a1a84d6eb3770f23367fbc30,cam_01,513,2025-12-06 15:57:19.067,513,6
2,store_01,c5c81de4a1a84d6eb3770f23367fbc30,cam_01,512,2025-12-06 15:57:19.066,512,6
3,store_01,c5c81de4a1a84d6eb3770f23367fbc30,cam_01,511,2025-12-06 15:57:19.064,511,6
4,store_01,c5c81de4a1a84d6eb3770f23367fbc30,cam_01,509,2025-12-06 15:57:19.063,509,6


In [9]:
df_bronze_sample = query("""
SELECT
    store_id,
    camera_id,
    frame_index,
    ingest_ts,
    json_extract_scalar(payload, '$.frame_index') AS frame_index_json
FROM lakehouse.rva.bronze_raw
ORDER BY ingest_ts DESC
LIMIT 10
""")
df_bronze_sample


🔍 Query: 
SELECT
    store_id,
    camera_id,
    frame_index,
    ingest_ts,
    json_extract_scalar(payload...


/tmp/ipykernel_118446/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 10 rows


,store_id,camera_id,frame_index,ingest_ts,frame_index_json
0,store_01,cam_01,3214,2025-12-06 16:00:17.670,3214
1,store_01,cam_01,3213,2025-12-06 16:00:17.670,3213
2,store_01,cam_01,3211,2025-12-06 16:00:17.669,3211
3,store_01,cam_01,3212,2025-12-06 16:00:17.669,3212
4,store_01,cam_01,3210,2025-12-06 16:00:17.668,3210
5,store_01,cam_01,3208,2025-12-06 16:00:17.667,3208
6,store_01,cam_01,3209,2025-12-06 16:00:17.667,3209
7,store_01,cam_01,3206,2025-12-06 16:00:17.666,3206
8,store_01,cam_01,3207,2025-12-06 16:00:17.666,3207
9,store_01,cam_01,3204,2025-12-06 16:00:17.665,3204


In [10]:
df_frame_stats = query("""
SELECT
    COUNT(*)                                     AS total_records,
    COUNT(*) FILTER (WHERE frame_index IS NULL)  AS null_frame_index,
    COUNT(*) FILTER (WHERE frame_index IS NOT NULL) AS non_null_frame_index
FROM lakehouse.rva.bronze_raw
""")
df_frame_stats


🔍 Query: 
SELECT
    COUNT(*)                                     AS total_records,
    COUNT(*) FILTER (WHER...


/tmp/ipykernel_118446/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 1 rows


,total_records,null_frame_index,non_null_frame_index
0,4114,0,4114


In [10]:
# View one full raw JSON event
df_raw = query("""
SELECT payload 
FROM lakehouse.rva.bronze_raw 
ORDER BY ingest_ts DESC 
LIMIT 1
""")

if df_raw is not None and len(df_raw) > 0:
    raw_json = json.loads(df_raw.iloc[0]['payload'])
    print("\n📄 Sample Bronze event structure:")
    print(json.dumps(raw_json, indent=2)[:1500])
    print("\n[...output truncated...]")

🔍 Query: 
SELECT payload 
FROM lakehouse.rva.bronze_raw 
ORDER BY ingest_ts DESC 
LIMIT 1



/tmp/ipykernel_215201/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 1 rows

📄 Sample Bronze event structure:
{
  "schema_version": "1.0",
  "pipeline_run_id": "c8eeb28b99e547b7b373fa4908a6cdca",
  "source": {
    "store_id": "store_01",
    "camera_id": "cam_01",
    "stream_id": "stream_01"
  },
  "frame_index": 9406,
  "capture_ts": "2025-12-06T12:33:25.433836+00:00",
  "image_size": {
    "width": 1920,
    "height": 1080
  },
  "detections": [
    {
      "det_id": "9406-0",
      "class": "person",
      "class_id": 0,
      "conf": 0.731907844543457,
      "bbox": {
        "x1": 948.5800170898438,
        "y1": 51.35393142700195,
        "x2": 1020.7776489257812,
        "y2": 237.62998962402344
      },
      "bbox_norm": {
        "x": 0.49405209223429364,
        "y": 0.04754993650648329,
        "w": 0.037602933247884114,
        "h": 0.1724778316639088
      },
      "centroid": {
        "x": 984,
        "y": 144
      },
      "centroid_norm": {
        "x": 0.5128535588582357,
        "y": 0.13378885233843768
      },
      "

### 📊 Bronze Data Quality Checks

In [11]:
# Check for NULL values
df_nulls = query("""
SELECT 
    COUNT(*) as total_records,
    COUNT(*) FILTER (WHERE payload IS NULL) as null_payload,
    COUNT(*) FILTER (WHERE store_id IS NULL) as null_store_id,
    COUNT(*) FILTER (WHERE camera_id IS NULL) as null_camera_id
FROM lakehouse.rva.bronze_raw
""")
print("\n🔍 NULL check:")
df_nulls

🔍 Query: 
SELECT 
    COUNT(*) as total_records,
    COUNT(*) FILTER (WHERE payload IS NULL) as null_payload,...
✅ Returned 1 rows

🔍 NULL check:


/tmp/ipykernel_215201/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


,total_records,null_payload,null_store_id,null_camera_id
0,9406,0,0,0


In [12]:
# Events per minute (last hour)
df_rate = query("""
SELECT 
    DATE_TRUNC('minute', ingest_ts) as minute,
    COUNT(*) as events_count
FROM lakehouse.rva.bronze_raw
WHERE ingest_ts >= CURRENT_TIMESTAMP - INTERVAL '1' HOUR
GROUP BY DATE_TRUNC('minute', ingest_ts)
ORDER BY minute DESC
LIMIT 10
""")
print("\n📈 Ingestion rate (events per minute - last hour):")
df_rate

🔍 Query: 
SELECT 
    DATE_TRUNC('minute', ingest_ts) as minute,
    COUNT(*) as events_count
FROM lakehouse....
✅ Returned 0 rows

📈 Ingestion rate (events per minute - last hour):


/tmp/ipykernel_215201/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


,minute,events_count


In [13]:
# Detections distribution
df_detections = query("""
SELECT 
    json_array_length(json_extract(payload, '$.detections')) as num_detections,
    COUNT(*) as count_events
FROM lakehouse.rva.bronze_raw
WHERE ingest_ts >= CURRENT_TIMESTAMP - INTERVAL '10' MINUTE
GROUP BY json_array_length(json_extract(payload, '$.detections'))
ORDER BY num_detections
""")
print("\n👥 Detections per frame distribution (last 10 min):")
df_detections

🔍 Query: 
SELECT 
    json_array_length(json_extract(payload, '$.detections')) as num_detections,
    COUNT(*...
✅ Returned 0 rows

👥 Detections per frame distribution (last 10 min):


/tmp/ipykernel_215201/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


,num_detections,count_events


---
## 📝 Summary

**Bronze Layer Status:**
- ✅ Table exists and accessible
- ✅ Data is being ingested
- ✅ JSON structure is valid

**Next Steps:**
- Explore Silver layer (cleaned detections)
- Explore Gold layer (aggregated metrics)
- Validate pipeline end-to-end